# Design of experiments (DOE): DPO & RLFT hyperparameter sweeps

This notebook is a **thin demo**: every step calls into the tested
`geap_tuning` package. See
[`docs/notes/doe-and-visualization.md`](../docs/notes/doe-and-visualization.md).

The *same* `run_sweep` driver as [`10_doe.ipynb`](10_doe.ipynb) runs **DPO** and
**RLFT** sweeps — `SweepConfig.method` (`"DPO"` / `"RLFT"`) selects the launcher
via the `_LAUNCHERS` registry (dispatching to `launch_preference_job` /
`launch_rlft_job`). Two things differ per method:

- **Headline metric.** DPO has no accuracy — its headline is `win_rate` (a
  base-model autorater: how often the tuned reply beats the dispreferred one).
  RLFT headlines `accuracy` (reward > 0 ⇒ correct). Read them from
  `HEADLINE_METRIC` / `METRICS_BY_METHOD` instead of hardcoding.
- **Non-scalar reward.** RLFT's `reward_config` is an object; carry it in
  `sweep.fixed` so it stays out of the run slug, the aggregate rows, and the
  Experiments params (`doe._scalar_params` filters it).

> **Requires live GCP and incurs tuning cost** (this launches ~8 tuning jobs).
> Have a real `.env` and `gcloud auth` in place; keep the tuning/Experiments
> region aligned. Charts need the optional viz group (`uv sync --group viz`).

In [ ]:
from geap_tuning.config import genai_client, load_config
from geap_tuning.doe import HEADLINE_METRIC, METRICS_BY_METHOD

cfg = load_config()
client = genai_client(cfg)  # tuning is regional-only
cfg

## Part A — DPO sweep

### A1. Build and stage the preference dataset

Same deterministic support-reply preference splits as the DPO notebook; the test
split is held out locally for the offline autorater.

In [ ]:
from geap_tuning.gcs import upload_file
from geap_tuning.preference.data import (
    SUPPORT_REPLIES,
    build_preference_dataset,
    build_preference_records,
    split_dataset,
)

paths = build_preference_dataset("../datasets/preference_support_style")
dpo_train_uri = upload_file(paths["train"], f"{cfg.bucket}/doe_dpo/train.jsonl")
dpo_val_uri = upload_file(paths["val"], f"{cfg.bucket}/doe_dpo/val.jsonl")

_, _, dpo_test_triples = split_dataset(SUPPORT_REPLIES)
dpo_test_records = build_preference_records(dpo_test_triples)
dpo_train_uri, dpo_val_uri

### A2. Declare the sweep and the autorater scorer

`method="DPO"` selects `launch_preference_job`; the grid crosses `beta` x
`epochs`. `run_sweep` calls `evaluate_fn(endpoint)` per run — here a blind A/B
judgment (tuned reply vs. the dispreferred reference) by a base model.

In [ ]:
from geap_tuning.doe import SweepConfig
from geap_tuning.experiments import init_experiment
from geap_tuning.inference import generate
from geap_tuning.preference.evaluate import run_preference_eval

DPO_EXPERIMENT = "geap-doe-dpo"
JUDGE_MODEL = "gemini-2.5-flash"
_JUDGE_PROMPT = (
    "You are judging two customer-support replies to the same message. Pick the "
    "reply that is warmer, more concise, and more helpful (acknowledges the issue "
    "and offers a next step). Answer with only the single letter 'A' or 'B'.\n\n"
    "Customer message: {user}\n\nReply A: {a}\n\nReply B: {b}\n\nBetter reply:"
)

dpo_sweep = SweepConfig(
    name="dpo",
    method="DPO",
    base_model="gemini-2.5-flash",  # flash-lite may not support preference tuning
    grid={"beta": [0.05, 0.1], "epochs": [1, 2]},  # 4 runs
)


def dpo_judge(user_text: str, cand_a: str, cand_b: str) -> str:
    verdict = generate(
        client, JUDGE_MODEL, _JUDGE_PROMPT.format(user=user_text, a=cand_a, b=cand_b)
    )
    return verdict[:1].upper()


def dpo_evaluate(endpoint: str) -> dict:
    return run_preference_eval(
        dpo_test_records,
        generate_fn=lambda user_text, e=endpoint: generate(client, e, user_text),
        judge_fn=dpo_judge,
    )


init_experiment(DPO_EXPERIMENT, project=cfg.project, location=cfg.location)

### A3. Run the DPO sweep, compare, and chart

Each grid point reuses a matching job if one exists (cost control), else launches
a DPO job. DPO's headline is `win_rate`; `plot_metric_bars` draws one bar per
run.

In [ ]:
from geap_tuning.doe import aggregate_results, run_sweep, select_best_run
from geap_tuning.viz import plot_metric_bars

dpo_results = run_sweep(
    client,
    dpo_sweep,
    train_uri=dpo_train_uri,
    val_uri=dpo_val_uri,
    evaluate_fn=dpo_evaluate,
    experiment=DPO_EXPERIMENT,
    labels=cfg.labels,
)
dpo_metric = HEADLINE_METRIC["DPO"]
dpo_rows = aggregate_results(dpo_results, metrics=METRICS_BY_METHOD["DPO"])
print(
    "best DPO run:",
    select_best_run({r.spec.name: r.metrics for r in dpo_results}, metric=dpo_metric),
)
plot_metric_bars(dpo_rows, metric=dpo_metric)

## Part B — RLFT sweep

### B1. Build the dataset and preflight the reward

The reward is a **declarative string-match** scorer (rewards an `Answer: <n>`
line) — cheap and deterministic, no sandbox. Preflight it once on a single record
via `validate_reward` before launching; RLFT auto-stops if >80% of reward calls
fail.

In [ ]:
from geap_tuning.rlft.data import (
    MATH_PROBLEMS,
    build_rlft_dataset,
    build_rlft_records,
)
from geap_tuning.rlft.data import split_dataset as split_math
from geap_tuning.rlft.tune import build_string_match_reward_config, validate_reward_config

paths = build_rlft_dataset("../datasets/rlft_math")
rlft_train_uri = upload_file(paths["train"], f"{cfg.bucket}/doe_rlft/train.jsonl")
rlft_val_uri = upload_file(paths["val"], f"{cfg.bucket}/doe_rlft/val.jsonl")

rlft_train_problems, _, rlft_test_problems = split_math(MATH_PROBLEMS)
rlft_train_records = build_rlft_records(rlft_train_problems)
rlft_test_records = build_rlft_records(rlft_test_problems)

reward = build_string_match_reward_config()
validate_reward_config(
    client,
    project=cfg.project,
    location=cfg.location,
    sample_answer="Answer: 4",
    example_record=rlft_train_records[0],
    reward_config=reward,
)

### B2. Declare the sweep (reward in `fixed`) and run it

`method="RLFT"` selects `launch_rlft_job`; the grid crosses `epochs` x
`samples_per_prompt`. The non-scalar `reward_config` rides in `sweep.fixed` — it
is splatted to the launcher but filtered out of the slug / rows / Experiments
params. RLFT headlines `accuracy`.

> **Base model** `gemini-3.5-flash` — verify region availability before running.
> The client stays regional (the global endpoint excludes tuning).

In [ ]:
from geap_tuning.rlft.evaluate import run_rlft_eval

RLFT_EXPERIMENT = "geap-doe-rlft"

rlft_sweep = SweepConfig(
    name="rlft",
    method="RLFT",
    base_model="gemini-3.5-flash",
    grid={"epochs": [2, 4], "samples_per_prompt": [4, 8]},  # 4 runs
    fixed={"reward_config": reward},
)


def rlft_evaluate(endpoint: str) -> dict:
    return run_rlft_eval(
        rlft_test_records,
        generate_fn=lambda user_text, e=endpoint: generate(client, e, user_text),
    )


init_experiment(RLFT_EXPERIMENT, project=cfg.project, location=cfg.location)
rlft_results = run_sweep(
    client,
    rlft_sweep,
    train_uri=rlft_train_uri,
    val_uri=rlft_val_uri,
    evaluate_fn=rlft_evaluate,
    experiment=RLFT_EXPERIMENT,
    labels=cfg.labels,
)
rlft_metric = HEADLINE_METRIC["RLFT"]
rlft_rows = aggregate_results(rlft_results, metrics=METRICS_BY_METHOD["RLFT"])
print(
    "best RLFT run:",
    select_best_run({r.spec.name: r.metrics for r in rlft_results}, metric=rlft_metric),
)
plot_metric_bars(rlft_rows, metric=rlft_metric)

## Read the tracked runs back from Experiments

`experiment_dataframe` returns a pandas table matching **Agent Platform Studio →
Experiments**. [`11_multi_run_viz.ipynb`](11_multi_run_viz.ipynb) charts either
experiment directly with **zero tuning cost**.

In [ ]:
from geap_tuning.experiments import experiment_dataframe

display(experiment_dataframe(DPO_EXPERIMENT))
experiment_dataframe(RLFT_EXPERIMENT)

## Next steps

Swap the RLFT reward (autorater, composite) via `sweep.fixed`, or read these runs
back with no tuning cost in [`11_multi_run_viz.ipynb`](11_multi_run_viz.ipynb)
(pass `--experiment geap-doe-dpo` / `geap-doe-rlft`). See
[`docs/notes/doe-and-visualization.md`](../docs/notes/doe-and-visualization.md).